# BUPA Liver Disorders - TabTransformer (PyTorch Tabular)
Compatible with Google Colab.

In [6]:
!pip -q install pytorch-tabular pandas scikit-learn

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from pytorch_tabular import TabularModel
from pytorch_tabular.models import TabTransformerConfig
from pytorch_tabular.config import DataConfig, TrainerConfig, OptimizerConfig


# ==========================
# LOAD DATASET BUPA
# ==========================

DATA_PATH='/content/drive/MyDrive/Clinical Reasoning Agent/liver+disorders/bupa.data'

cols=[
    'mcv',
    'alkphos',
    'sgpt',
    'sgot',
    'gammagt',
    'drinks',
    'selector'
]

df=pd.read_csv(DATA_PATH,names=cols)


# Target 0/1
df['selector']=df['selector'].astype(int)-1


# IMPORTANT : continuous -> float
continuous_cols=[
    'mcv',
    'alkphos',
    'sgpt',
    'sgot',
    'gammagt',
    'drinks'
]

df[continuous_cols]=df[continuous_cols].astype(float)


# ==========================
# SPLIT
# ==========================

train,test=train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['selector']
)



# ==========================
# CONFIG
# ==========================

data_config=DataConfig(
    target=['selector'],
    continuous_cols=continuous_cols,
    categorical_cols=[]
)


trainer_config=TrainerConfig(
    max_epochs=50,
    batch_size=64,
    accelerator="auto"
)


optimizer_config=OptimizerConfig()


model_config=TabTransformerConfig(
    task="classification",
    learning_rate=1e-3
)



# ==========================
# MODEL
# ==========================

model=TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config
)



# ==========================
# TRAIN
# ==========================

model.fit(
    train=train,
    validation=test
)



# ==========================
# PREDICTION
# ==========================

pred=model.predict(test)

print(pred.head())
print(pred.columns)



# récupérer automatiquement la colonne prediction
prediction_col=[c for c in pred.columns if "prediction" in c.lower()]

if len(prediction_col)==0:
    raise Exception("Aucune colonne prediction trouvée")

y_pred=pred[prediction_col[0]].astype(int).values


y_true=test['selector'].values



# ==========================
# RESULTS
# ==========================

print("\nAccuracy:")
print(accuracy_score(y_true,y_pred))


print("\nClassification report:")
print(classification_report(y_true,y_pred))


print("\nConfusion matrix:")
print(confusion_matrix(y_true,y_pred))



# ==========================
# SAVE
# ==========================

model.save_model(
    "tabtransformer_bupa"
)

print("\nModel saved successfully")

Epoch 49/49 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 0:00:00 • 0:00:00 43.02it/s v_num: 0.000 train_loss: 0.598      
                                                                               valid_loss: 0.668 valid_accuracy:   
                                                                               0.565 train_accuracy: 0.576         

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.


2026-07-22 11:29:45,753 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-07-22 11:29:45,754 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model
INFO:pytorch_lightning.trainer.connectors.checkpoint_connector:`weights_only` was not set, defaulting to `False`.


     selector_0_probability  selector_1_probability  selector_prediction
57                 0.380857                0.619143                    1
160                0.679889                0.320111                    0
183                0.169552                0.830448                    1
144                0.518412                0.481588                    0
292                0.455148                0.544852                    1
Index(['selector_0_probability', 'selector_1_probability',
       'selector_prediction'],
      dtype='object')

Accuracy:
0.5652173913043478

Classification report:
              precision    recall  f1-score   support

           0       0.48      0.52      0.50        29
           1       0.63      0.60      0.62        40

    accuracy                           0.57        69
   macro avg       0.56      0.56      0.56        69
weighted avg       0.57      0.57      0.57        69


Confusion matrix:
[[15 14]
 [16 24]]

Model saved successfully


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
